## Import the necessary libraries

In [1]:
import os
import json
import math
import random
from pathlib import Path
import numpy as np
from PIL import Image

## Setting the seed for reproducability

In [2]:
def set_seed(seed):
  random.seed(seed)
  np.random.seed(seed)

seed = 42
set_seed(seed)

## Function for Generating Images and Masks

In [3]:
LABELS = {
    0: "background",
    1: "white_matter",
    2: "gray_matter",
    3: "cortex",
    4: "CSF",
}

def create_grid(height, width, rot_angle=0.0, translation_x=0.0, translation_y=0.0):
  """
    This function creates a grid for training, rotation test and translation test sets.
    By using grids, we can apply the changes easier.
  """
  yy, xx = np.meshgrid(np.arange(height), np.arange(width), indexing="ij")

  cx = (width-1)/2.0
  cy = (height-1)/2.0

  # Moving origin to the image center and undoing the translation
  x = xx-cx-translation_x
  y = yy-cy-translation_y

  # Undoing rotation
  theta = math.radians(rot_angle)
  cos = math.cos(theta)
  sin = math.sin(theta)

  x0 = cos*x + sin*y
  y0 = -sin*x + cos*y

  return x0, y0

def generate_sample_and_mask(height, width, angle=0.0, translation_x=0.0, translation_y=0.0):
  """
    Function to generate the images and their corresponding masks. This function is used for
    generating images and masks of rotated test set, translated test set and training sets.

    If angle, translation_x and translation_y is 0, then it generated training set.
    If only angle is provided, then it generates rotated images & masks.
    If only translation_x and translation_y are given, it generates translated images & masks.
  """

  # PART 1: GENERATING THE SAMPLE IMAGE

  # Have a clear x and y coordinates
  x, y = create_grid(height=height, width=width, rot_angle=angle, translation_x=translation_x, translation_y=translation_y)

  # Generate the main brain structure
  # We set the values by hand because we want to have similar size of brain-like ellipses
  # to make a fair comparison
  outer_brain_x = np.random.uniform(24.0,27.0)
  outer_brain_y = np.random.uniform(28.0,31.0)

  # Outer brain structure
  outer_brain = (x/outer_brain_x)**2 + (y/outer_brain_y)**2 <= 1.0

  # White matter
  white_x = outer_brain_x * np.random.uniform(0.50,0.60)
  white_y = outer_brain_y * np.random.uniform(0.50,0.60)
  white_matter = (x / white_x) ** 2 + (y / white_y) ** 2 <= 1.0

  # Gray matter
  gray_x = outer_brain_x * np.random.uniform(0.78,0.85)
  gray_y = outer_brain_y * np.random.uniform(0.78,0.85)
  gray_matter = (x / gray_x) ** 2 + (y / gray_y) ** 2 <= 1.0

  # CSF
  csf_radius = np.random.uniform(3.0, 4.5)
  csf_y = np.random.uniform(-2.0,2.0)
  csf_x = np.random.uniform(6.0,8.5)

  left_csf = ((x + csf_x) ** 2 + (y - csf_y) ** 2) <= csf_radius**2

  right_csf = ((x - csf_x) ** 2 + (y - csf_y) ** 2) <= csf_radius**2

  # Choose one of the CSF
  csf = left_csf | right_csf

  # PART 2: GENERATING THE MASK
  mask = np.zeros((height, width), dtype=np.uint8)

  # Add labels to the mask ordered by their size
  mask[outer_brain] = 3
  mask[gray_matter] = 2
  mask[white_matter] = 1
  mask[csf] = 4

  # PART 3: CREATE IMAGE
  image = np.zeros((height, width), dtype=np.float32)

  # Approximate MRI-like intensities
  image[mask == 0] = np.random.normal(0.05, 0.015, size=(mask == 0).sum())
  image[mask == 1] = np.random.normal(0.70, 0.060, size=(mask == 1).sum())
  image[mask == 2] = np.random.normal(0.52, 0.060, size=(mask == 2).sum())
  image[mask == 3] = np.random.normal(0.45, 0.060, size=(mask == 3).sum())
  image[mask == 4] = np.random.normal(0.25, 0.050, size=(mask == 4).sum())

  return image, mask


## Function for applying transformations

In [4]:
def transform(transformation):
  """
    Function to choose rotation and translation according to the transformation name.
  """

  angle = 0.0
  translation_x = 0.0
  translation_y = 0.0

  if transformation == "rotated":
    sign = np.random.choice([-1,1])
    angle = sign * np.random.uniform(5.0, 15.0) # Same rotation angle from the paper
  elif transformation == "translation":
    translation_x_sign =  np.random.choice([-1,1])
    translation_y_sign = np.random.choice([-1,1])

    translation_x = translation_x_sign * np.random.uniform(5.0,10.0) # Same translation from the paper
    translation_y = translation_y_sign * np.random.uniform(5.0,10.0)
  elif transformation == "rotated_translation":
    sign = np.random.choice([-1,1])
    translation_x_sign =  np.random.choice([-1,1])
    translation_y_sign = np.random.choice([-1,1])

    angle = sign * np.random.uniform(5.0, 15.0)
    translation_x = translation_x_sign * np.random.uniform(5.0,10.0)
    translation_y = translation_y_sign * np.random.uniform(5.0,10.0)

  return angle, translation_x, translation_y


## Visualization & Saving functions

In [5]:
def save_color_masks(mask, mask_path):
  height, width = mask.shape
  colored = np.zeros((height,width,3), dtype=np.uint8)

  colored[mask == 0] = [0, 0, 0]
  colored[mask == 1] = [120, 120, 255]
  colored[mask == 2] = [255, 120, 120]
  colored[mask == 3] = [120, 255, 120]
  colored[mask == 4] = [100,255, 90]

  Image.fromarray(colored).save(mask_path)

def save_label_mask(mask, mask_path):
    Image.fromarray(mask.astype(np.uint8)).save(mask_path)

## Generating datasets

In [6]:
def generate_dataset(output_dir, height, width):
  root = Path(output_dir)

  split_sizes = {
      "train": 400,
      "validation": 100,
      "test": 125,
      "rotated": 125,
      "translation": 125,
      "rotated_translation": 125
  }

  # Create directories for splits
  for split in split_sizes.keys():
    (root / split / "images").mkdir(parents=True, exist_ok=True)
    (root / split / "masks").mkdir(parents=True, exist_ok=True)
    (root / split / "masks_color").mkdir(parents=True, exist_ok=True)

  # Generate and save images and masks
  for split_name, n_samples in split_sizes.items():
    for i in range(n_samples):
      angle, translation_x, translation_y = transform(split_name)

      image, mask = generate_sample_and_mask(height=height, width=width, angle=angle,
                                             translation_x=translation_x, translation_y=translation_y)

      # Normalize and convert to uint8 for saving
      image_uint8 = np.clip(image, 0.0, 1.0)
      image_uint8 = (image_uint8 * 255).astype(np.uint8)

      image_path = root / split_name / "images" / f"{split_name}_{i:04d}.png"
      mask_path = root / split_name / "masks" / f"{split_name}_{i:04d}.png"

      Image.fromarray(image_uint8).save(image_path)
      save_label_mask(mask, mask_path)

      color_mask_path = root / split_name / "masks_color" / f"{split_name}_{i:04d}.png"
      save_color_masks(mask, color_mask_path)

    print(f"Dataset {split_name} is generated!")


In [7]:
generate_dataset("/content/toy-dataset", 192, 192)

Dataset train is generated!
Dataset validation is generated!
Dataset test is generated!
Dataset rotated is generated!
Dataset translation is generated!
Dataset rotated_translation is generated!
